In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import json
load_dotenv(override=True)

In [ ]:
if os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("OpenAI API key is set correctly and begins with 'sk-'.")

In [ ]:
openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
import sqlite3

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"Price has been set to {price}"

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:


get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name" : "set_ticket_price",
    "description" : "set return ticket price for a destination city",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type": "string",
                "description" : "the destination city for which the return ticket price is to be set"
            },
            "return_ticket_price" : {
                "type" : "number",
                "description" : "the return ticket price in dollars"
            }
        },
        "required" : ["destination_city", "return_ticket_price"],
        "additionalProperties" : False
    }
}

In [ ]:
tools = [{"type" : "function" , "function" : get_price_function}, {"type" : "function" , "function" : set_price_function}]

In [ ]:
def handle_tool_calls(message):
    responses = []
    tool_calls = message.tool_calls
    for tool_call in tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role" : "tool" , 
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price = arguments.get("return_ticket_price")
            price_details = set_ticket_price(city, price)
            responses.append({
                "role" : "tool" , 
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
    return responses



In [ ]:
def airline_bot(message, history):
    history = [{"role" : h["role"], "content" : h["content"]} for h in history]
    messages = [{"role" : "system", "content" : system_message}] + history + [{"role" : "user", "content" : message}]
    response = openai.chat.completions.create(model=MODEL, messages  = messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        response = handle_tool_calls(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content
    

In [ ]:
gr.ChatInterface(fn=airline_bot, type="messages").launch(inbrowser=True)